<a href="https://colab.research.google.com/github/Swingandamiss/ML_projects/blob/main/podium_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
import pandas as pd
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')
df = pd.merge(results, races[['raceId', 'year', 'round', 'circuitId', 'name']], on='raceId', how='left')
print(df.head())

   resultId  raceId  driverId  constructorId number  grid position  \
0         1      18         1              1     22     1        1   
1         2      18         2              2      3     5        2   
2         3      18         3              3      7     7        3   
3         4      18         4              4      5    11        4   
4         5      18         5              1     23     3        5   

  positionText  positionOrder  points  ...  milliseconds fastestLap rank  \
0            1              1    10.0  ...       5690616         39    2   
1            2              2     8.0  ...       5696094         41    3   
2            3              3     6.0  ...       5698779         41    5   
3            4              4     5.0  ...       5707797         58    7   
4            5              5     4.0  ...       5708630         43    1   

  fastestLapTime fastestLapSpeed statusId  year  round  circuitId  \
0       1:27.452         218.300        1  2008      

In [77]:
drivers = pd.read_csv('drivers.csv')
constructors = pd.read_csv('constructors.csv')
status = pd.read_csv('status.csv')
df = pd.merge(df, drivers[['driverId', 'driverRef', 'dob']], on='driverId', how='left')
constructors = constructors.rename(columns={'name': 'constructor_name'})
df = pd.merge(df, constructors[['constructorId', 'constructor_name']], on='constructorId', how='left')
df = pd.merge(df, status[['statusId', 'status']], on='statusId', how='left')
# Clean up the columns for readability
df = df.rename(columns={'name': 'race_name', 'driverRef': 'driver_name'})
print(f"Dataset size: {df.shape[0]} rows, {df.shape[1]} columns")
print(df[['year', 'race_name', 'driver_name', 'constructor_name', 'grid', 'positionOrder', 'status']].head(10))

Dataset size: 26759 rows, 26 columns
   year              race_name driver_name constructor_name  grid  \
0  2008  Australian Grand Prix    hamilton          McLaren     1   
1  2008  Australian Grand Prix    heidfeld       BMW Sauber     5   
2  2008  Australian Grand Prix     rosberg         Williams     7   
3  2008  Australian Grand Prix      alonso          Renault    11   
4  2008  Australian Grand Prix  kovalainen          McLaren     3   
5  2008  Australian Grand Prix    nakajima         Williams    13   
6  2008  Australian Grand Prix    bourdais       Toro Rosso    17   
7  2008  Australian Grand Prix   raikkonen          Ferrari    15   
8  2008  Australian Grand Prix      kubica       BMW Sauber     2   
9  2008  Australian Grand Prix       glock           Toyota    18   

   positionOrder     status  
0              1   Finished  
1              2   Finished  
2              3   Finished  
3              4   Finished  
4              5   Finished  
5              6     +1

In [78]:
# Filter out the ancient history
df = df[df['year'] >= 2014].copy()

# Create our Target Variable: The Podium!
df['podium'] = df['positionOrder'].apply(lambda x: 1 if x <= 3 else 0)

# Sort the data chronologically so it makes sense
df = df.sort_values(by=['year', 'round', 'positionOrder'])

print("Total modern races to train on:", df['raceId'].nunique())
print(df[['year', 'driver_name', 'constructor_name', 'positionOrder', 'podium']].head(10))

Total modern races to train on: 228
       year      driver_name constructor_name  positionOrder  podium
22127  2014          rosberg         Mercedes              1       1
22128  2014  kevin_magnussen          McLaren              2       1
22129  2014           button          McLaren              3       1
22130  2014           alonso          Ferrari              4       0
22131  2014           bottas         Williams              5       0
22132  2014       hulkenberg      Force India              6       0
22133  2014        raikkonen          Ferrari              7       0
22134  2014           vergne       Toro Rosso              8       0
22135  2014            kvyat       Toro Rosso              9       0
22136  2014            perez      Force India             10       0


In [79]:
# Sort the data by driver, then chronologically by year and round
df = df.sort_values(by=['driverId', 'year', 'round'])

# Calculate the rolling 3-race average of points for each driver
df['driver_form'] = df.groupby('driverId')['points'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# If a driver is a rookie (or it's race 1), they won't have a past 3 races. Fill those blanks with 0.
df['driver_form'] = df['driver_form'].fillna(0)

# Let's check our work for a specific driver, like Max Verstappen (driverId 830)
max_data = df[df['driver_name'] == 'hamilton'][['year', 'race_name', 'points', 'driver_form']]
print(max_data.tail(10))

       year                 race_name  points  driver_form
26566  2024          Dutch Grand Prix     4.0    21.666667
26583  2024        Italian Grand Prix    10.0    14.666667
26607  2024     Azerbaijan Grand Prix     2.0    13.000000
26624  2024      Singapore Grand Prix     8.0     5.333333
26658  2024  United States Grand Prix     0.0     6.666667
26662  2024    Mexico City Grand Prix    12.0     3.333333
26688  2024      São Paulo Grand Prix     1.0     6.666667
26700  2024      Las Vegas Grand Prix    18.0     4.333333
26730  2024          Qatar Grand Prix     0.0    10.333333
26742  2024      Abu Dhabi Grand Prix    12.0     6.333333


In [80]:
print(df['driver_name'].unique())

['hamilton' 'rosberg' 'alonso' 'raikkonen' 'kubica' 'massa' 'sutil'
 'button' 'vettel' 'grosjean' 'kobayashi' 'hulkenberg' 'maldonado' 'resta'
 'perez' 'ricciardo' 'vergne' 'chilton' 'gutierrez' 'bottas'
 'jules_bianchi' 'kevin_magnussen' 'kvyat' 'lotterer' 'ericsson' 'stevens'
 'max_verstappen' 'nasr' 'sainz' 'merhi' 'rossi' 'jolyon_palmer'
 'wehrlein' 'haryanto' 'vandoorne' 'ocon' 'stroll' 'giovinazzi' 'gasly'
 'brendon_hartley' 'leclerc' 'sirotkin' 'norris' 'russell' 'albon'
 'latifi' 'pietro_fittipaldi' 'aitken' 'tsunoda' 'mazepin'
 'mick_schumacher' 'zhou' 'de_vries' 'piastri' 'sargeant' 'lawson'
 'bearman' 'colapinto' 'doohan']


In [81]:
from sklearn.metrics import accuracy_score, confusion_matrix

season_2023_df = test_df[test_df['year'] == 2023].copy()

def predict_driver_podiums_2023():
    driver_name = input("Enter a driver name (e.g., max_verstappen, hamilton, alonso): ").strip().lower()

    # Filter the 2023 data for the specific driver
    driver_2023_data = season_2023_df[season_2023_df['driver_name'] == driver_name].copy()

    if driver_2023_data.empty:
        print(f"No 2023 data found for driver: '{driver_name}'. Please check the spelling.")
        print("Available drivers in 2023:", season_2023_df['driver_name'].unique())
        return

    # Isolate the features and actual target for prediction
    X_driver = driver_2023_data[features]
    y_true = driver_2023_data['podium']

    # Make predictions using the previously trained XGBoost model
    predictions = model.predict(X_driver)

    # Add predictions back to the dataframe to visualize the results
    driver_2023_data['predicted_podium'] = predictions

    # Select columns to display
    display_cols = ['race_name', 'constructor_name', 'grid', 'driver_form', 'predicted_podium', 'podium']
    results_display = driver_2023_data[display_cols].rename(columns={'podium': 'actual_podium'})

    # Print the race-by-race results
    print(f"\n--- 2023 Podium Predictions for {driver_name} ---")

    # Map the binary outputs to readable text
    results_display['predicted_podium'] = results_display['predicted_podium'].map({1: 'Yes', 0: 'No'})
    results_display['actual_podium'] = results_display['actual_podium'].map({1: 'Yes', 0: 'No'})

    # Reset index for a cleaner print output
    print(results_display.reset_index(drop=True).to_string())

    # Calculate performance metrics
    acc = accuracy_score(y_true, predictions)
    cm = confusion_matrix(y_true, predictions, labels=[0, 1])

    # Print the metrics
    print(f"\n--- Performance Metrics for {driver_name} (2023) ---")
    print(f"Accuracy: {acc:.2f}")
    print("Confusion Matrix:")
    print(cm)
    print("\nMatrix Breakdown:")
    print(f"True Negatives (Correctly predicted No Podium):  {cm[0][0]}")
    print(f"False Positives (Incorrectly predicted Podium):  {cm[0][1]}")
    print(f"False Negatives (Incorrectly predicted No Podium): {cm[1][0]}")
    print(f"True Positives  (Correctly predicted Podium):    {cm[1][1]}")
predict_driver_podiums_2023()

Enter a driver name (e.g., max_verstappen, hamilton, alonso): max_verstappen

--- 2023 Podium Predictions for max_verstappen ---
                   race_name constructor_name  grid  driver_form predicted_podium actual_podium
0         Bahrain Grand Prix         Red Bull     1    19.333333              Yes           Yes
1   Saudi Arabian Grand Prix         Red Bull    15    19.333333               No           Yes
2      Australian Grand Prix         Red Bull     1    23.000000              Yes           Yes
3      Azerbaijan Grand Prix         Red Bull     2    23.000000              Yes           Yes
4           Miami Grand Prix         Red Bull     9    20.666667              Yes           Yes
5          Monaco Grand Prix         Red Bull     1    23.000000              Yes           Yes
6         Spanish Grand Prix         Red Bull     1    23.000000              Yes           Yes
7        Canadian Grand Prix         Red Bull     1    25.666667              Yes           Yes
8      